# Predicción de bajas – FV8 (HT4960-FV8)

**FV8 = FV4 con el feature engineering, los canaritos y la limpieza nuevos** (ver `modificaciones_proximo_experimento.md`). El final es el de FV4: limpieza cruzada de los 4 meses + un ensamble de 10 recetas.

| # | Paso | Meses | Modelos |
|---|---|---|---|
| 1 | Se eliminan los préstamos personales; `ccajas_depositos` de 202105 → NA | todos | — |
| 2 | **Percentil por mes** para todas las originales (0 → 0; positivos en (0, 1]; negativos en [−1, 0)), menos 12 excepciones que quedan en crudo | todos | — |
| 3 | Lags 1–3, diferencias 1–3 y acumulado **sobre los percentiles** (y sobre las 12 excepciones) | todos | — |
| 3 bis | **Script del docente:** tendencia (`_tend3`) y promedio (`_avg3`) de 3 meses sobre las mismas variables; se juzgan en el canarito 2 | todos | — |
| 4 | **Ronda 1:** todas contra todas entre las originales (×, ÷, ÷ invertido) | solo 202104 | — |
| 5 | **Canarito 1:** juez = mejor receta de FV4, entrenado con 202104 completo, importancia por **Gain**, 1 semilla, corte media + 2 desvíos | 202104 (control: 202106) | 5 |
| 6 | **Ronda 2:** las 20 mejores sobrevivientes (ejes) contra las sobrevivientes que no participaron de la ronda 1 | solo 202104 | — |
| 7 | **Canarito 2** (mismo juez) → `campos_finales` | 202104 (control: 202106) | 5 |
| 8 | Las interacciones de `campos_finales` se calculan en todos los meses; se borran las originales en crudo | todos | — |
| 9 | Bayesiana: **se reusa `BO_log.txt` de FV4** (solo corre si faltan iteraciones; en ese caso, antes limpia con el modelo30 del docente) | — | — |
| 10 | **Prueba:** limpieza cruzada A/B de 202103–04 + 10 recetas, predice 202106 | 202103 + 202104 | 20 × 2 + 10 |
| 11 | **Final (como FV4):** limpieza cruzada A/B de los 4 meses + 10 recetas, predice 202108 | 202103 a 202106 | 20 × 2 + 10 |
| 12 | 15 archivos de entrega | 202108 | — |

- `_lag2`, `_lag3`, `_delta2`, `_delta3` son 100% NA en 202104: **no pasan por el canarito y se conservan siempre**.
- Los canaritos se sortean una vez por filtro (la matriz del juez se arma una sola vez, para no duplicar memoria); las 5 semillas cambian el modelo.
- **Checkpoints:** cada semilla de cada canarito, los filtros, `campos_finales`, las limpiezas, la prueba y la predicción final se guardan. Si la VM se corta, se relanza **el mismo comando** y retoma.
- **FV9** (la versión con el final de FV5) va a copiar de acá los canaritos, `campos_finales`, la limpieza de la prueba y la prueba.

Runtime: **R**.

## 0. Clase ternaria

In [ ]:
require( "data.table" )

dataset <- fread("~/buckets/b1/datasets/competencia_01_crudo.csv" )

dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]

setorder( dsimple, numero_de_cliente, periodo0 )

periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1

dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]

setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file =  "~/buckets/b1/datasets/competencia_01.csv.gz",
    sep = ","
)

dataset[, .N, keyby = list(foto_mes, clase_ternaria)]

## 1. Librerías

In [ ]:
rm(list = ls(all.names = TRUE))
gc(verbose = FALSE)

paquetes <- c("data.table", "R.utils", "rlist",
              "bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging",
              "lightgbm", "Rcpp")
faltantes <- paquetes[!paquetes %in% rownames(installed.packages())]
if (length(faltantes) > 0) install.packages(faltantes)

library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
library(DiceKriging)
library(lightgbm)
library(Rcpp)

setDTthreads(0)   # todos los núcleos de la VM
format(Sys.time(), "%a %b %d %X %Y")

## 2. Parámetros
Todo lo ajustable está acá. Las semillas se generan solas a partir de la primigenia (la primera de la lista es la primigenia). **Misma primigenia que FV4/FV5**, para que FV9 pueda reusar lo que calcula FV8.

In [ ]:
PARAM <- list()
PARAM$experimento <- "HT4960-FV8"
PARAM$bo_log_origen <- "HT4960-FV4"   # de acá se copia BO_log.txt (recetas y juez de los canaritos)
PARAM$semilla_primigenia <- 379531

# rutas
PARAM$carpeta_base <- path.expand("~/buckets/b1")
PARAM$dataset <- file.path(PARAM$carpeta_base, "datasets", "competencia_01.csv.gz")

# meses
PARAM$train_prueba  <- c(202103, 202104)
PARAM$future_prueba <- c(202106)
PARAM$train_final   <- c(202103, 202104, 202105, 202106)
PARAM$future_final  <- c(202108)
PARAM$mes_solo_historia <- c(202107)   # solo da historia (lags) a 202108

# feature engineering: variables que quedan en crudo (sin percentil)
PARAM$excepciones <- c("cliente_vip", "cliente_edad", "cliente_antiguedad", "cproductos",
                       "ctarjeta_visa", "ctarjeta_master",
                       "Master_delinquency", "Master_status", "Master_Finiciomora",
                       "Visa_delinquency", "Visa_status", "Visa_Finiciomora")

# canaritos (juez nuevo)
PARAM$juez_mes_train     <- 202104   # el juez entrena con este mes completo
PARAM$juez_mes_control   <- 202106   # solo control: AUC del juez
PARAM$juez_filas_control <- 50000    # muestra de 202106 para el control (todas las bajas + CONTINUA)
PARAM$juez_semillas      <- 1   # (era 5) 1 semilla por tiempo: entrega el domingo
PARAM$canaritos_pct      <- 0.20
PARAM$n_ejes             <- 20       # ronda 2: mejores sobrevivientes del canarito 1
PARAM$regex_forzadas     <- "_(lag2|lag3|delta2|delta3)$"   # 100% NA en 202104: se conservan siempre

# modelo30 (limpiezas)
PARAM$training_pct    <- 70     # partición 70/30 de 202103+202104 (limpieza de la bayesiana)
PARAM$n_semillas30    <- 20
PARAM$undersampling30 <- 0.10   # % de CONTINUA que usa el modelo30
PARAM$umbral_prob30   <- 0.5    # CONTINUA con prob30 mayor a esto se sacan

# bayesiana (solo corre si BO_log.txt tiene menos iteraciones que esto)
PARAM$bo_iteraciones <- 50
PARAM$bo_semillas    <- 5

# ensamble de recetas (prueba y final)
PARAM$n_recetas           <- 5    # (era 10) por tiempo: entrega el domingo
PARAM$hacer_prueba        <- FALSE   # (por tiempo) sin prueba sobre 202106: se compara con la ganancia pública
PARAM$semillas_por_receta <- 1

# ganancia y entrega
PARAM$ganancia_acierto <- 1072500
PARAM$ganancia_error   <- -27500
PARAM$cortes <- seq(8000, 15000, by = 500)

PARAM$num_threads <- parallel::detectCores()   # se adapta a la VM

# pool de semillas
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillas <- unique(c(PARAM$semilla_primigenia, sample(100000:999999, 200)))[1:100]

In [ ]:
# parámetros fijos del LightGBM (la bayesiana pisa num_iterations, num_leaves y min_sum_hessian_in_leaf)
PARAM$lgbm$param_fijos <- list(
  boosting= "gbdt",
  objective= "binary",
  metric= "auc",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,
  num_threads= PARAM$num_threads,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,
  feature_fraction= 0.50,
  learning_rate= 0.005,

  num_iterations= 20000,
  num_leaves= 1024,
  min_sum_hessian_in_leaf= 1e-06
)

# modelo30 (parámetros del docente)
PARAM$lgbm$param30 <- modifyList(PARAM$lgbm$param_fijos, list(
  num_iterations= 2180L,
  num_leaves= 30L,
  min_sum_hessian_in_leaf= 0.1141634103
))

PARAM$hs <- ps(
  num_iterations= p_int(lower= 64L,  upper= 4096L),
  num_leaves= p_int(lower= 16L,  upper= 2048L),
  min_sum_hessian_in_leaf= p_dbl(lower= 1e-06, upper= 0.1)
)

In [ ]:
# ---------- funciones auxiliares ----------
# reloj: imprime y además escribe en log_FV8.txt (nbconvert no muestra nada hasta terminar)
memoria_vm <- function() {
  m <- tryCatch(system("free -g | awk 'NR==2{print $3\"/\"$2}'", intern = TRUE), error = function(e) "?")
  paste0("RAM ", m, " GB")
}
reloj <- function(msg) {
  linea <- paste(format(Sys.time(), "%d/%m %H:%M:%S"), "-", msg, "-", memoria_vm())
  cat(linea, "\n")
  arch <- file.path(PARAM$carpeta_base, "exp", PARAM$experimento, "log_FV8.txt")
  if (dir.exists(dirname(arch))) cat(linea, "\n", file = arch, append = TRUE)
}

safe_div <- function(a, b) {
  r <- a / b
  r[!is.na(b) & b == 0] <- 0
  r
}

# AUC calculada por rangos
auc <- function(y, p) {
  r <- frank(p, ties.method = "average")
  n1 <- sum(y == 1); n0 <- sum(y == 0)
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

loguear <- function(reg, arch) {
  if (!file.exists(arch)) {
    cat(paste0("fecha\t", paste(names(reg), collapse = "\t"), "\n"), file = arch)
  }
  valores <- sapply(reg, function(x) paste(x, collapse = "_"))
  linea <- paste0(format(Sys.time(), "%Y%m%d.%H%M%S"), "\t", paste(valores, collapse = "\t"), "\n")
  cat(linea, file = arch, append = TRUE)
  cat(linea)
}

# divide filas en 2 mitades, estratificado por clase
partir_mitades <- function(idx, semilla) {
  set.seed(semilla, kind = "L'Ecuyer-CMRG")
  cl <- dataset$clase_ternaria[idx]
  mitad <- integer(length(idx))
  for (k in unique(cl)) {
    pos <- which(cl == k)
    mitad[pos] <- sample(rep(1:2, length.out = length(pos)))
  }
  mitad
}

NO_FEATURES <- c("clase_ternaria", "clase01", "fold", "prob30", "aporta", "aporta_prueba", "aporta_final")

# ---------- interacciones ----------
# Ronda 1: A__x__B, A__div__B   (A y B son originales)
# Ronda 2: E__X2__C, E__DIV2__C (E y C pueden ser cualquier variable, incluso de la ronda 1)
# El nombre codifica la fórmula: cualquier interacción se puede recalcular en cualquier mes.
OPS <- c("__X2__", "__DIV2__", "__x__", "__div__")   # primero los de la ronda 2

valor_col <- function(nm, idx, operando) {
  for (op in OPS) {
    if (grepl(op, nm, fixed = TRUE)) {
      p <- strsplit(nm, op, fixed = TRUE)[[1]]
      if (length(p) != 2) stop("Nombre de interacción inválido: ", nm)
      a <- operando(p[1]); b <- operando(p[2])
      return(if (op %in% c("__X2__", "__x__")) a * b else safe_div(a, b))
    }
  }
  if (!nm %in% names(dataset)) stop("Columna desconocida: ", nm)
  as.numeric(dataset[[nm]][idx])
}

# devuelve una función que calcula (y recuerda) los operandos para las filas idx
nuevo_operando <- function(idx) {
  cache <- new.env(hash = TRUE)
  operando <- function(nm) {
    v <- cache[[nm]]
    if (is.null(v)) {
      v <- valor_col(nm, idx, operando)
      assign(nm, v, envir = cache)
    }
    v
  }
  operando
}

# matriz numérica (filas idx) con las columnas pedidas + canaritos uniformes, sin copias intermedias
armar_matriz <- function(campos, idx, nombres_can = character(0), semilla_can = NULL) {
  operando <- nuevo_operando(idx)
  X <- matrix(NA_real_, nrow = length(idx), ncol = length(campos) + length(nombres_can),
              dimnames = list(NULL, c(campos, nombres_can)))
  for (j in seq_along(campos)) {
    X[, j] <- valor_col(campos[j], idx, operando)
    if (j %% 5000 == 0) reloj(paste("  columnas", j, "/", length(campos)))
  }
  if (length(nombres_can) > 0) {
    set.seed(semilla_can, kind = "L'Ecuyer-CMRG")
    for (k in seq_along(nombres_can)) X[, length(campos) + k] <- runif(length(idx))
  }
  rm(operando); gc(verbose = FALSE)
  X
}

tipo_de <- function(v) {
  fifelse(grepl("^canarito_", v), "canarito",
  fifelse(grepl("__X2__|__DIV2__", v), "ronda2",
  fifelse(grepl("__x__|__div__", v), "ronda1",
  fifelse(grepl("_(tend3|avg3)$", v), "docente", "base"))))
}

# ---------- checkpoints ----------
ck <- function(nombre) file.path("checkpoints", nombre)
guardar_ck <- function(dt, nombre) {
  tmp <- file.path("checkpoints", paste0("tmp_", nombre))   # misma extensión: fwrite comprime igual
  fwrite(dt, tmp)
  file.rename(tmp, ck(nombre))
  cat("Checkpoint guardado:", nombre, "\n")
}
ids_texto <- function(x) if (is.character(x)) x else format(x, scientific = FALSE, trim = TRUE)
clave <- function(cli, mes) paste(ids_texto(cli), mes)
leer_ck <- function(nombre) fread(ck(nombre), colClasses = list(character = "numero_de_cliente"))
posiciones_ck <- function(dt_ck) {
  pos <- match(clave(dt_ck$numero_de_cliente, dt_ck$foto_mes),
               clave(dataset$numero_de_cliente, dataset$foto_mes))
  if (anyNA(pos)) stop("El checkpoint no coincide con el dataset")
  pos
}

In [ ]:
carpeta_exp <- file.path(PARAM$carpeta_base, "exp", PARAM$experimento)
dir.create(carpeta_exp, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_exp)
dir.create("entregas", showWarnings = FALSE)
dir.create("checkpoints", showWarnings = FALSE)
cat("Carpeta de trabajo:", getwd(), "\n")

# BO_log.txt de FV4: recetas del ensamble y receta del juez de los canaritos
if (!file.exists("BO_log.txt")) {
  origen_bo <- file.path(PARAM$carpeta_base, "exp", PARAM$bo_log_origen, "BO_log.txt")
  if (!file.exists(origen_bo)) stop("No se encontró ", origen_bo)
  file.copy(origen_bo, "BO_log.txt")
  cat("Copiado BO_log.txt desde", PARAM$bo_log_origen, "\n")
}
tb_bo <- fread("BO_log.txt")
setorder(tb_bo, -metrica)
receta_juez <- tb_bo[1, list(num_iterations = as.integer(num_iterations),
                             num_leaves = as.integer(num_leaves),
                             min_sum_hessian_in_leaf = as.double(min_sum_hessian_in_leaf))]
PARAM$lgbm$param_juez <- modifyList(PARAM$lgbm$param_fijos, c(as.list(receta_juez),
  list(force_row_wise = FALSE, force_col_wise = TRUE)))   # col-wise: menos memoria con muchas columnas
reloj(paste("Comienza", PARAM$experimento, "-", PARAM$num_threads, "núcleos"))
cat("Juez de los canaritos (mejor receta de", PARAM$bo_log_origen, "):\n")
print(tb_bo[1, list(num_iterations, num_leaves, min_sum_hessian_in_leaf, metrica)])

## 3. Feature engineering
### 3.1 Lectura y correcciones

In [ ]:
reloj("Leyendo el dataset")
dataset <- fread(PARAM$dataset)
dataset[, c("cprestamos_personales", "mprestamos_personales") := NULL]

n_ceros <- dataset[foto_mes == 202105 & ccajas_depositos == 0, .N]
dataset[foto_mes == 202105, ccajas_depositos := NA]
cat("ccajas_depositos en 202105:", n_ceros, "ceros pasados a NA\n")

ORIGINALES <- setdiff(names(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria"))
no_num <- ORIGINALES[!sapply(dataset[, ORIGINALES, with = FALSE], is.numeric)]
if (length(no_num) > 0) stop(paste("Variables no numéricas:", paste(no_num, collapse = ", ")))
faltan <- setdiff(PARAM$excepciones, ORIGINALES)
if (length(faltan) > 0) stop(paste("Excepciones no encontradas:", paste(faltan, collapse = ", ")))
if (any(grepl("__", ORIGINALES, fixed = TRUE))) stop("Hay originales con '__' en el nombre")

CON_PCT <- setdiff(ORIGINALES, PARAM$excepciones)   # llevan percentil; en crudo se borran al final
cat("Originales:", length(ORIGINALES), " con percentil:", length(CON_PCT),
    " excepciones (en crudo):", length(PARAM$excepciones), "\n")

### 3.2 Percentil por mes (regla nueva)
Dentro de cada `foto_mes`: **0 → 0**; positivos equidistantes en **(0, 1]** (el más grande vale 1); negativos equidistantes en **[−1, 0)** (el más negativo vale −1); NA → NA. Empates: puesto promedio. Columna nueva: `<variable>_pct`.

In [ ]:
pct_signo <- function(x) {
  r <- rep(NA_real_, length(x))
  r[!is.na(x) & x == 0] <- 0
  p <- which(x > 0)
  if (length(p) > 0) r[p] <- frank(x[p], ties.method = "average") / length(p)
  n <- which(x < 0)
  if (length(n) > 0) r[n] <- -frank(-x[n], ties.method = "average") / length(n)
  r
}

reloj("Calculando percentiles")
for (col in CON_PCT) {
  dataset[, (paste0(col, "_pct")) := pct_signo(get(col)), by = foto_mes]
}
VARS_PCT <- paste0(CON_PCT, "_pct")
cat("Percentiles creados:", length(VARS_PCT), "\n")

# control: rango de algunos percentiles
print(rbindlist(lapply(head(VARS_PCT, 8), function(v)
  data.table(variable = v, min = min(dataset[[v]], na.rm = TRUE), max = max(dataset[[v]], na.rm = TRUE),
             ceros = sum(dataset[[v]] == 0, na.rm = TRUE), na = sum(is.na(dataset[[v]]))))))

### 3.3 Lags, diferencias y acumulado
Sobre los percentiles y las 12 excepciones (las originales en crudo no llevan derivadas). Siempre diferencias, nunca tasas.

In [ ]:
vars_lag <- c(VARS_PCT, PARAM$excepciones)
cat("Variables a las que se les calculan lags:", length(vars_lag), "\n")

setorder(dataset, numero_de_cliente, foto_mes)
cliente <- dataset$numero_de_cliente
periodo <- (dataset$foto_mes %/% 100L) * 12L + dataset$foto_mes %% 100L

valido <- lapply(1:3, function(k) {
  cl <- shift(cliente, k)
  pe <- shift(periodo, k)
  !is.na(cl) & cl == cliente & pe == periodo - k
})
pos_primera <- match(cliente, cliente)
es_primera  <- pos_primera == seq_along(cliente)

reloj("Calculando lags, diferencias y acumulado")
for (i in seq_along(vars_lag)) {
  v <- vars_lag[i]
  x <- as.numeric(dataset[[v]])

  l1 <- shift(x, 1); l1[!valido[[1]]] <- NA
  l2 <- shift(x, 2); l2[!valido[[2]]] <- NA
  l3 <- shift(x, 3); l3[!valido[[3]]] <- NA
  acum <- x - x[pos_primera]; acum[es_primera] <- NA

  dataset[, (paste0(v, "_lag1"))   := l1]
  dataset[, (paste0(v, "_lag2"))   := l2]
  dataset[, (paste0(v, "_lag3"))   := l3]
  dataset[, (paste0(v, "_delta1")) := x - l1]
  dataset[, (paste0(v, "_delta2")) := l1 - l2]
  dataset[, (paste0(v, "_delta3")) := l2 - l3]
  dataset[, (paste0(v, "_acum"))   := acum]

  if (i %% 25 == 0) reloj(paste(i, "/", length(vars_lag)))
}
rm(cliente, periodo, valido, pos_primera, es_primera, x, l1, l2, l3, acum)
gc(verbose = FALSE)
cat("Columnas después de los lags:", ncol(dataset), "\n")

### 3.3 bis Tendencia y promedio de 3 meses (script del docente)
Código del docente, sin cambios en las funciones (`fhistC` en C vía Rcpp + `TendenciaYmuchomas`). Se aplica a las mismas variables que los lags (percentiles + 12 excepciones), con `ventana = 3`, `tendencia = TRUE` y `promedio = TRUE` → columnas `_tend3` y `_avg3`.
- Se calcula **antes de sacar 202107**, para que la ventana de 202108 use 202106–202107–202108.
- En 202104 la ventana solo tiene 202103 y 202104 (los datos empiezan en 202103); en 202103 quedan NA.
- **No pasan por el canarito 1** (así se reusa la semilla ya calculada): se juzgan recién en el **canarito 2**, junto con las sobrevivientes del canarito 1 y la ronda 2. No son ejes ni "contra" de la ronda 2.

In [ ]:
cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")
#------------------------------------------------------------------------------

# calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos
# La funcionalidad de ratioavg es autoria de  Daiana Sparta,  UAustral  2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}
#------------------------------------------------------------------------------
#------------------------------------------------------------------------------

reloj("Tendencia y promedio de 3 meses (docente)")
setorderv(dataset, c("numero_de_cliente", "foto_mes"))
TendenciaYmuchomas(dataset,
  cols= vars_lag,
  ventana= 3,
  tendencia= TRUE,
  minimo= FALSE,
  maximo= FALSE,
  promedio= TRUE,
  ratioavg= FALSE,
  ratiomax= FALSE
)
VARS_DOCENTE <- c(paste0(vars_lag, "_tend3"), paste0(vars_lag, "_avg3"))
if (!all(VARS_DOCENTE %in% names(dataset))) stop("Faltan columnas _tend3/_avg3")
reloj(paste("Variables del docente creadas:", length(VARS_DOCENTE)))

### 3.4 Preparación
Se saca 202107 (solo daba historia a 202108), se crea `clase01` y la partición 70/30 de 202103+202104 (la usa la limpieza de la bayesiana, si se corre). Se definen los grupos de variables.

In [ ]:
dataset <- dataset[!(foto_mes %in% PARAM$mes_solo_historia)]
gc(verbose = FALSE)

dataset[, clase01 := as.integer(clase_ternaria %in% c("BAJA+1", "BAJA+2"))]

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
bloque <- c(rep(1L, PARAM$training_pct), rep(2L, 100L - PARAM$training_pct))
dataset[, fold := NA_integer_]
dataset[foto_mes %in% PARAM$train_prueba,
        fold := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
        by = clase_ternaria]

# grupos de variables
BASE <- setdiff(names(dataset), c(NO_FEATURES, CON_PCT, VARS_DOCENTE, "numero_de_cliente", "foto_mes"))
FORZADAS <- c("numero_de_cliente", "foto_mes", grep(PARAM$regex_forzadas, BASE, value = TRUE))
JUZGABLES_BASE <- setdiff(BASE, FORZADAS)

pares1 <- combn(ORIGINALES, 2, simplify = FALSE)
RONDA1 <- unlist(lapply(pares1, function(p)
  c(paste0(p[1], "__x__", p[2]), paste0(p[1], "__div__", p[2]), paste0(p[2], "__div__", p[1]))))

cat("Filas:", nrow(dataset), " Columnas:", ncol(dataset), "\n")
cat("Base (variables del modelo antes de interacciones):", length(BASE), "\n")
cat("  forzadas (no pasan por el canarito):", length(FORZADAS), "\n")
cat("  juzgadas por el canarito 1:", length(JUZGABLES_BASE), "\n")
cat("Ronda 1 (todas contra todas entre", length(ORIGINALES), "originales):", length(RONDA1), "columnas\n")
dataset[, .N, keyby = foto_mes]

## 4. Canarito (función)
1. Arma **una matriz con 202104 completo**: las variables a juzgar + 20% de canaritos uniformes (las interacciones se calculan solo para estas filas).
2. Juez = **mejor receta de FV4**, sin undersampling. Se entrena con **5 semillas** sobre el mismo `lgb.Dataset`.
3. Importancia = **Gain** de LightGBM (`lgb.importance`). Puesto 1 = más Gain; las no usadas (Gain 0) van al último puesto.
4. **Control:** AUC de cada semilla sobre una muestra de 202106 (todas las bajas + CONTINUA hasta 50.000). No interviene en el corte.
5. Se promedia el puesto entre las 5 semillas y se corta en **media + 2 desvíos** de los puestos de los canaritos. Sobreviven las variables reales por encima del corte.

Cada semilla se guarda en `checkpoints/`; si la VM se corta, solo se rehacen las que faltan.

In [ ]:
correr_canarito <- function(etapa, campos) {
  arch <- sprintf("filtro_canaritos_etapa%d.csv", etapa)
  if (file.exists(arch)) {
    reloj(paste("Checkpoint: canarito", etapa, "ya estaba hecho"))
    return(fread(arch))
  }
  n_can <- ceiling(PARAM$canaritos_pct * length(campos))
  nombres_can <- sprintf("canarito_%05d", seq_len(n_can))
  semillas <- PARAM$semillas[seq_len(PARAM$juez_semillas)]
  arch_sem <- function(s) ck(sprintf("canarito%d_semilla_%d.csv", etapa, s))
  hechas <- semillas[file.exists(sapply(semillas, arch_sem))]
  for (s in hechas) {   # una semilla guardada tiene que ser de esta misma lista de variables
    if (!setequal(fread(arch_sem(s), select = "variable")$variable, c(campos, nombres_can)))
      stop("El checkpoint ", arch_sem(s), " es de otra lista de variables: borrá checkpoints/canarito",
           etapa, "_semilla_*.csv y relanzá")
  }
  pendientes <- setdiff(semillas, hechas)
  cat("Canarito", etapa, "- variables reales:", length(campos), " canaritos:", n_can,
      " semillas pendientes:", length(pendientes), "\n")

  if (length(pendientes) > 0) {
    idx_tr <- dataset[, which(foto_mes == PARAM$juez_mes_train)]
    reloj(paste("Armando la matriz de", PARAM$juez_mes_train))
    X <- armar_matriz(campos, idx_tr, nombres_can, PARAM$semilla_primigenia + 10 * etapa)
    reloj(paste("Matriz:", nrow(X), "x", ncol(X), "-", round(8 * as.numeric(nrow(X)) * ncol(X) / 2^30, 1), "GB"))
    dtrain <- lgb.Dataset(X, label = dataset$clase01[idx_tr], free_raw_data = TRUE,
                          params = modifyList(PARAM$lgbm$param_juez, list(seed = pendientes[1])))
    lgb.Dataset.construct(dtrain)
    rm(X); gc(verbose = FALSE)
    reloj("lgb.Dataset construido")

    # control: muestra de 202106
    idx_c <- dataset[, which(foto_mes == PARAM$juez_mes_control)]
    set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
    pos <- idx_c[dataset$clase01[idx_c] == 1L]
    neg <- idx_c[dataset$clase01[idx_c] == 0L]
    idx_c <- c(pos, sample(neg, max(0, min(length(neg), PARAM$juez_filas_control - length(pos)))))
    Xc <- armar_matriz(campos, idx_c, nombres_can, PARAM$semilla_primigenia + 10 * etapa + 1)
    yc <- dataset$clase01[idx_c]

    for (s in pendientes) {
      t0 <- Sys.time()
      modelo <- lgb.train(data = dtrain, params = modifyList(PARAM$lgbm$param_juez, list(seed = s)))
      auc_c <- auc(yc, predict(modelo, Xc))
      imp <- lgb.importance(modelo)
      tb_s <- data.table(variable = c(campos, nombres_can))
      tb_s <- merge(tb_s, imp[, list(variable = Feature, gain = Gain)], by = "variable", all.x = TRUE)
      tb_s[is.na(gain), gain := 0]
      setorder(tb_s, -gain)
      n_total <- nrow(tb_s)
      tb_s[, puesto := as.numeric(seq_len(.N))]
      tb_s[gain == 0, puesto := as.numeric(n_total)]
      tb_s[, `:=`(semilla = s, auc_control = auc_c)]
      guardar_ck(tb_s, basename(arch_sem(s)))
      rm(modelo, imp); gc(verbose = FALSE)
      reloj(sprintf("Canarito %d: semilla %d lista (%.1f min) - AUC control %d = %.4f",
                    etapa, s, as.numeric(difftime(Sys.time(), t0, units = "mins")),
                    PARAM$juez_mes_control, auc_c))
    }
    rm(dtrain, Xc); gc(verbose = FALSE)
  }

  rankings <- rbindlist(lapply(semillas, function(s) fread(arch_sem(s))))
  print(rankings[, list(auc_control = auc_control[1], usadas = sum(gain > 0)), by = semilla])

  tb <- rankings[, list(puesto = mean(puesto), gain = mean(gain)), by = variable]
  tb[, canarito := variable %in% nombres_can]
  tb[, tipo := tipo_de(variable)]
  setorder(tb, puesto)
  corte <- tb[canarito == TRUE, mean(puesto) + 2 * sd(puesto)]
  tb[, sobrevive := !canarito & puesto < corte]
  fwrite(tb, arch)

  mejor_can <- tb[canarito == TRUE, min(puesto)]
  cat("\nCanarito", etapa, "- corte (media + 2 desvíos):", round(corte, 1),
      " de", nrow(tb), "puestos. Mejor canarito:", round(mejor_can, 1), "\n")
  cat("Reales mejores que todos los canaritos:", tb[canarito == FALSE & puesto < mejor_can, .N], "\n")
  print(tb[canarito == FALSE, list(reales = .N, sobreviven = sum(sobrevive),
                                   pct = round(100 * mean(sobrevive), 1)), keyby = tipo])
  tb
}

## 5. Canarito 1
Juzga: percentiles + derivadas (sin las forzadas) + las 12 excepciones + **la ronda 1 completa** (todas contra todas entre las originales, calculada solo en 202104).

In [ ]:
tabla1 <- correr_canarito(1, c(JUZGABLES_BASE, RONDA1))
sobrev1 <- tabla1[sobrevive == TRUE][order(puesto), variable]
reloj(paste("Sobrevivientes del canarito 1:", length(sobrev1)))
print(tabla1[1:30, list(puesto, variable, gain, tipo)])

## 6. Ronda 2 de interacciones
- **Ejes:** las 20 mejores sobrevivientes del canarito 1, de cualquier tipo.
- **Contra:** las sobrevivientes que **no participaron de la ronda 1** (ni interacciones de la ronda 1 ni originales), más los ejes entre sí.
- Para cada par: eje × otra, eje ÷ otra, otra ÷ eje. Cada par se genera una sola vez y se saltean los pares de originales (ya probados en la ronda 1). Sin filtros de redundancia.

In [ ]:
ejes <- head(sobrev1, PARAM$n_ejes)
contra <- sobrev1[!grepl("__x__|__div__", sobrev1) & !(sobrev1 %in% ORIGINALES)]
candidatos <- unique(c(contra, ejes))

vistos <- new.env(hash = TRUE)
RONDA2 <- vector("list", length(ejes))
for (i in seq_along(ejes)) {
  e <- ejes[i]
  nuevas <- character(0)
  for (c2 in candidatos) {
    if (c2 == e) next
    if (e %in% ORIGINALES && c2 %in% ORIGINALES) next   # ya probado en la ronda 1
    k <- paste(sort(c(e, c2)), collapse = "|")
    if (exists(k, envir = vistos, inherits = FALSE)) next
    assign(k, TRUE, envir = vistos)
    nuevas <- c(nuevas, paste0(e, "__X2__", c2), paste0(e, "__DIV2__", c2), paste0(c2, "__DIV2__", e))
  }
  RONDA2[[i]] <- nuevas
}
RONDA2 <- unlist(RONDA2)
rm(vistos)

reloj(paste("Ronda 2 - ejes:", length(ejes), " contra:", length(contra), " columnas nuevas:", length(RONDA2)))
print(data.table(eje = ejes, tipo = tipo_de(ejes)))

## 7. Canarito 2 → `campos_finales`
Juzga las sobrevivientes del canarito 1 + las variables del docente (`_tend3`, `_avg3`) + la ronda 2. `campos_finales` = sobrevivientes del canarito 2 + forzadas (`numero_de_cliente`, `foto_mes` y todas las `_lag2`, `_lag3`, `_delta2`, `_delta3`).

In [ ]:
tabla2 <- correr_canarito(2, c(sobrev1, VARS_DOCENTE, RONDA2))
sobrev2 <- tabla2[sobrevive == TRUE][order(puesto), variable]

if (file.exists("campos_finales.csv")) {
  campos_finales <- fread("campos_finales.csv")$variable
  cat("Checkpoint: campos_finales.csv recuperado\n")
} else {
  campos_finales <- c(FORZADAS, sobrev2)
  fwrite(data.table(variable = campos_finales), "campos_finales.csv")
}
reloj(paste("Variables finales:", length(campos_finales)))
print(table(tipo_de(campos_finales)))
print(tabla2[1:30, list(puesto, variable, gain, tipo)])

## 8. Interacciones finales en todos los meses
Las interacciones de `campos_finales` (de las dos rondas) se calculan para todas las filas a partir de su nombre. Después se borran las originales en crudo (menos las 12 excepciones) y todo lo que no quedó en `campos_finales`.

In [ ]:
inter_finales <- grep("__x__|__div__|__X2__|__DIV2__", campos_finales, value = TRUE)
reloj(paste("Calculando", length(inter_finales), "interacciones en todas las filas"))
operando <- nuevo_operando(seq_len(nrow(dataset)))
for (nm in inter_finales) dataset[, (nm) := valor_col(nm, seq_len(nrow(dataset)), operando)]
rm(operando); gc(verbose = FALSE)

if (!all(campos_finales %in% names(dataset))) stop("Faltan variables de campos_finales en el dataset")
sobran <- setdiff(names(dataset), c(campos_finales, NO_FEATURES))
dataset[, (sobran) := NULL]
gc(verbose = FALSE)
cat("Columnas eliminadas:", length(sobran), " Quedan:", ncol(dataset), "\n")

## 9. Bayesiana (se reusa la de FV4)
Si `BO_log.txt` ya tiene las 50 iteraciones, **no corre nada** y se pasa a las recetas. Si faltaran iteraciones, primero limpia (modelo30 del docente, 20 semillas, sin canaritos, 10% de CONTINUA: el 30% juzga al 70% y se sacan los CONTINUA con prob30 > 0,5) y después corre las que faltan (AUC, 5 semillas).

In [ ]:
n_previas <- nrow(fread("BO_log.txt"))
bo_completa <- n_previas >= PARAM$bo_iteraciones
cat("Iteraciones previas:", n_previas, "- bayesiana completa:", bo_completa, "\n")

if (!bo_completa) {
  # ---- limpieza para la bayesiana (modelo30 del docente, 20 semillas, sin canaritos) ----
  if (file.exists(ck("aporta.csv.gz"))) {
    ck_ap <- leer_ck("aporta.csv.gz")
    pos <- posiciones_ck(ck_ap)
    dataset[, aporta := 0L]
    dataset[pos, aporta := ck_ap$aporta]
  } else {
    idx_f2 <- dataset[, which(foto_mes %in% PARAM$train_prueba & fold == 2L)]
    idx_f1 <- dataset[, which(foto_mes %in% PARAM$train_prueba & fold == 1L)]
    X_f1 <- data.matrix(dataset[idx_f1, campos_finales, with = FALSE])
    prob <- numeric(length(idx_f1))
    semillas30 <- PARAM$semillas[seq_len(PARAM$n_semillas30)]
    for (s in semillas30) {
      set.seed(s, kind = "L'Ecuyer-CMRG")
      az <- runif(length(idx_f2))
      idx_tr <- idx_f2[az <= PARAM$undersampling30 | dataset$clase01[idx_f2] == 1L]
      modelo <- lgb.train(
        data = lgb.Dataset(data.matrix(dataset[idx_tr, campos_finales, with = FALSE]),
                           label = dataset$clase01[idx_tr]),
        params = modifyList(PARAM$lgbm$param30, list(seed = s)))
      prob <- prob + predict(modelo, X_f1)
      rm(modelo)
    }
    prob <- prob / length(semillas30)
    rm(X_f1); gc(verbose = FALSE)
    dataset[, prob30 := NA_real_]
    dataset[idx_f1, prob30 := prob]
    dataset[, aporta := 0L]
    dataset[fold == 1L & (clase01 == 1L | prob30 <= PARAM$umbral_prob30), aporta := 1L]
    idx_ck <- dataset[, which(foto_mes %in% PARAM$train_prueba)]
    guardar_ck(dataset[idx_ck, list(numero_de_cliente = ids_texto(numero_de_cliente),
                                    foto_mes, aporta)], "aporta.csv.gz")
  }
  print(dataset[fold == 1L, .N, keyby = list(foto_mes, clase_ternaria, aporta)])

  # ---- bayesiana ----
  idx_bo_train <- dataset[, which(foto_mes %in% PARAM$train_prueba & fold == 1L & aporta == 1L)]
  idx_bo_valid <- dataset[, which(foto_mes %in% PARAM$train_prueba & fold == 2L)]
  X_bo_train <- data.matrix(dataset[idx_bo_train, campos_finales, with = FALSE])
  y_bo_train <- dataset$clase01[idx_bo_train]
  X_bo_valid <- data.matrix(dataset[idx_bo_valid, campos_finales, with = FALSE])
  y_bo_valid <- dataset$clase01[idx_bo_valid]
  dtrain_bo <- lgb.Dataset(X_bo_train, label = y_bo_train, free_raw_data = FALSE)

  giter <- fread("BO_log.txt")[, max(iter)]

  Estimar_lightgbm <- function(xs) {
    t0 <- Sys.time()
    giter <<- giter + 1
    param <- modifyList(PARAM$lgbm$param_fijos, xs)
    semillas_bo <- PARAM$semillas[seq_len(PARAM$bo_semillas)]
    aucs <- c()
    for (s in semillas_bo) {
      param$seed <- s
      modelo <- lgb.train(data = dtrain_bo, params = param)
      aucs <- c(aucs, auc(y_bo_valid, predict(modelo, X_bo_valid)))
      rm(modelo)
    }
    xx <- param
    xx$seed <- paste(semillas_bo, collapse = "_")
    xx$iter <- giter
    xx$metrica <- mean(aucs)
    loguear(xx, "BO_log.txt")
    cat("  iteración", giter, "-", round(difftime(Sys.time(), t0, units = "mins"), 1), "minutos\n")
    set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
    return(list(metrica = xx$metrica))
  }

  objective <- ObjectiveRFun$new(
    fun= Estimar_lightgbm,
    domain= PARAM$hs,
    codomain= ps(metrica = p_dbl(tags = "maximize")),
    id= "BO_lightgbm"
  )
  instance <- OptimInstanceBatchSingleCrit$new(
    objective= objective,
    terminator= trm("evals", n_evals = PARAM$bo_iteraciones)
  )
  optimizer <- opt("mbo",
    loop_function = bayesopt_ego,
    surrogate= srlrn(lrn("regr.km", optim.method = "BFGS", nugget.estim = TRUE,
                         control= list(trace = FALSE))),
    acq_function= acqf("ei"),
    acq_optimizer= acqo(
      optimizer  = opt("random_search", batch_size = 1000),
      terminator = trm("evals", n_evals = 1000))
  )
  tb_BO_previo <- fread("BO_log.txt")
  if (nrow(tb_BO_previo) > 0) {
    instance$archive$add_evals(
      xdt = tb_BO_previo[, list(num_iterations = as.integer(num_iterations),
                                num_leaves = as.integer(num_leaves),
                                min_sum_hessian_in_leaf = as.double(min_sum_hessian_in_leaf))],
      ydt = tb_BO_previo[, list(metrica = as.double(metrica))])
  }

  reloj("Comienza la bayesiana")
  bayesiana_salida <- optimizer$optimize(instance)
  reloj("Terminó la bayesiana")
  rm(X_bo_train, X_bo_valid, dtrain_bo); gc(verbose = FALSE)
} else {
  cat("Bayesiana completa: se usan las recetas de BO_log.txt (FV4)\n")
}

In [ ]:
# las 10 mejores recetas de BO_log.txt
tb_bayesiana <- fread("BO_log.txt")
setorder(tb_bayesiana, -metrica)
recetas <- head(unique(tb_bayesiana[, list(num_iterations = as.integer(num_iterations),
                                           num_leaves = as.integer(num_leaves),
                                           min_sum_hessian_in_leaf = as.double(min_sum_hessian_in_leaf),
                                           metrica)]), PARAM$n_recetas)
recetas[, receta := .I]
fwrite(recetas, "recetas.csv")
print(recetas)

## 10. Limpieza cruzada y ensamble de recetas (funciones)

**`limpiar_cruzado(meses)`:** divide las filas de esos meses en dos mitades (estratificadas por clase). Un modelo30 entrenado con la mitad 1 (20 semillas, 10% de CONTINUA) juzga a la mitad 2, y viceversa. Los CONTINUA con `prob30` promedio > 0,5 quedan afuera. Las bajas se conservan todas. Usa `campos_finales`, sin canaritos.

**`entrenar_ensamble(idx_train, idx_pred)`:** entrena un modelo por receta con las filas indicadas, sin undersampling, y promedia las probabilidades sobre `idx_pred`.

In [ ]:
limpiar_cruzado <- function(meses) {
  idx <- dataset[, which(foto_mes %in% meses)]
  mitad <- partir_mitades(idx, PARAM$semilla_primigenia + 1)
  prob <- numeric(length(idx))
  semillas30 <- PARAM$semillas[seq_len(PARAM$n_semillas30)]

  for (h in 1:2) {
    idx_eval <- idx[mitad == h]
    idx_otra <- idx[mitad != h]
    X_eval <- data.matrix(dataset[idx_eval, campos_finales, with = FALSE])
    acum <- numeric(length(idx_eval))
    for (i in seq_along(semillas30)) {
      s <- semillas30[i]
      set.seed(s, kind = "L'Ecuyer-CMRG")
      az <- runif(length(idx_otra))
      idx_tr <- idx_otra[az <= PARAM$undersampling30 | dataset$clase01[idx_otra] == 1L]
      modelo <- lgb.train(
        data = lgb.Dataset(data.matrix(dataset[idx_tr, campos_finales, with = FALSE]),
                           label = dataset$clase01[idx_tr]),
        params = modifyList(PARAM$lgbm$param30, list(seed = s)))
      acum <- acum + predict(modelo, X_eval)
      rm(modelo)
    }
    prob[mitad == h] <- acum / length(semillas30)
    rm(X_eval); gc(verbose = FALSE)
    reloj(paste("Limpieza cruzada: mitad", h, "lista"))
  }

  aporta <- integer(nrow(dataset))
  aporta[idx] <- as.integer(dataset$clase01[idx] == 1L | prob <= PARAM$umbral_prob30)
  aporta
}

entrenar_ensamble <- function(idx_train, idx_pred, label_col = "clase01") {
  X_train <- data.matrix(dataset[idx_train, campos_finales, with = FALSE])
  dtrain <- lgb.Dataset(X_train, label = dataset[[label_col]][idx_train], free_raw_data = FALSE)
  rm(X_train)
  X_pred <- data.matrix(dataset[idx_pred, campos_finales, with = FALSE])

  pred <- numeric(nrow(X_pred))
  n_mod <- 0
  for (r in seq_len(nrow(recetas))) {
    for (k in seq_len(PARAM$semillas_por_receta)) {
      n_mod <- n_mod + 1
      param <- modifyList(PARAM$lgbm$param_fijos, list(
        num_iterations = recetas$num_iterations[r],
        num_leaves = recetas$num_leaves[r],
        min_sum_hessian_in_leaf = recetas$min_sum_hessian_in_leaf[r],
        seed = PARAM$semillas[n_mod]))
      modelo <- lgb.train(data = dtrain, params = param)
      pred <- pred + predict(modelo, X_pred)
      rm(modelo); gc(verbose = FALSE)
      reloj(paste("Receta", r, "semilla", k, "lista"))
    }
  }
  pred / n_mod
}

## 11. Prueba sobre 202106 (SALTEADA por tiempo: `PARAM$hacer_prueba = FALSE`)
Si se activa: limpieza cruzada de 202103+202104 y ensamble de recetas entrenado con los dos meses juntos; mide la ganancia en 202106.

In [ ]:
if (PARAM$hacer_prueba) {
  if (file.exists(ck("aporta_prueba.csv.gz"))) {
    reloj("Checkpoint: se recupera la limpieza cruzada de la prueba")
    ck_p <- leer_ck("aporta_prueba.csv.gz")
    pos <- posiciones_ck(ck_p)
    dataset[, aporta_prueba := 0L]
    dataset[pos, aporta_prueba := ck_p$aporta_prueba]
  } else {
    dataset[, aporta_prueba := limpiar_cruzado(PARAM$train_prueba)]
    idx_ck <- dataset[, which(foto_mes %in% PARAM$train_prueba)]
    guardar_ck(dataset[idx_ck, list(numero_de_cliente = ids_texto(numero_de_cliente),
                                    foto_mes, aporta_prueba)], "aporta_prueba.csv.gz")
  }
  dataset[foto_mes %in% PARAM$train_prueba, .N, keyby = list(foto_mes, clase_ternaria, aporta_prueba)]
  
  if (file.exists("ganancia_prueba_202106.csv")) {
    cat("Checkpoint: la prueba ya estaba hecha\n")
    print(fread("ganancia_prueba_202106.csv"))
  } else {
    idx_tr_prueba <- dataset[, which(foto_mes %in% PARAM$train_prueba & aporta_prueba == 1L)]
    idx_prueba    <- dataset[, which(foto_mes %in% PARAM$future_prueba)]
    cat("Entrenamiento de la prueba:", length(idx_tr_prueba), "filas\n")
  
    pred_prueba <- entrenar_ensamble(idx_tr_prueba, idx_prueba)
  
    tb_prueba <- dataset[idx_prueba, list(numero_de_cliente, clase_ternaria)]
    tb_prueba[, prob := pred_prueba]
    tb_prueba[, gan := ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia_acierto, PARAM$ganancia_error)]
    setorder(tb_prueba, -prob)
    tb_prueba[, ganancia_acumulada := cumsum(gan)]
  
    tabla_ganancia <- data.table(envios = PARAM$cortes,
                                 ganancia = tb_prueba$ganancia_acumulada[PARAM$cortes])
    fwrite(tabla_ganancia[, list(envios, ganancia = format(ganancia, scientific = FALSE))],
           "ganancia_prueba_202106.csv")
  
    cat("\nEnvíos\tGanancia\n")
    for (k in seq_len(nrow(tabla_ganancia))) {
      cat(tabla_ganancia$envios[k], "\t", format(tabla_ganancia$ganancia[k], big.mark = ".", scientific = FALSE), "\n")
    }
    cat("\nMejor corte:", tabla_ganancia[which.max(ganancia), envios], "envíos\n")
  }
} else {
  reloj("Prueba sobre 202106 salteada (PARAM$hacer_prueba = FALSE)")
}

## 12. Modelo final (como FV4: entrena 202103–202106 juntos, predice 202108)
Limpieza cruzada de los 4 meses y ensamble de 10 recetas con todas las filas limpias.

In [ ]:
if (file.exists(ck("aporta_final.csv.gz"))) {
  reloj("Checkpoint: se recupera la limpieza cruzada del final")
  ck_f <- leer_ck("aporta_final.csv.gz")
  pos <- posiciones_ck(ck_f)
  dataset[, aporta_final := 0L]
  dataset[pos, aporta_final := ck_f$aporta_final]
} else {
  dataset[, aporta_final := limpiar_cruzado(PARAM$train_final)]
  idx_ck <- dataset[, which(foto_mes %in% PARAM$train_final)]
  guardar_ck(dataset[idx_ck, list(numero_de_cliente = ids_texto(numero_de_cliente),
                                  foto_mes, aporta_final)], "aporta_final.csv.gz")
}
dataset[foto_mes %in% PARAM$train_final, .N, keyby = list(foto_mes, clase_ternaria, aporta_final)]

if (file.exists("prediccion_202108.csv")) {
  cat("Checkpoint: la predicción final ya estaba hecha\n")
  tb_entrega <- fread("prediccion_202108.csv", colClasses = list(character = "numero_de_cliente"))
} else {
  idx_tr_final <- dataset[, which(foto_mes %in% PARAM$train_final & aporta_final == 1L)]
  idx_entrega  <- dataset[, which(foto_mes %in% PARAM$future_final)]
  cat("Entrenamiento final:", length(idx_tr_final), "filas\n")

  pred_final <- entrenar_ensamble(idx_tr_final, idx_entrega)

  tb_entrega <- dataset[idx_entrega, list(numero_de_cliente = ids_texto(numero_de_cliente))]
  tb_entrega[, prob := pred_final]
  setorder(tb_entrega, -prob)
  fwrite(tb_entrega, "prediccion_202108.csv.tmp")
  file.rename("prediccion_202108.csv.tmp", "prediccion_202108.csv")
}

## 13. Archivos para la entrega
15 archivos (8.000 a 15.000 cada 500), una sola columna `numero_de_cliente`, sin encabezado y con ids enteros. Cada archivo se relee y se verifica.

In [ ]:
ids <- ids_texto(tb_entrega$numero_de_cliente)

for (n in PARAM$cortes) {
  archivo <- file.path("entregas", sprintf("%s_%05d.csv", PARAM$experimento, n))
  fwrite(data.table(ids[1:n]), archivo, col.names = FALSE, quote = FALSE)

  lineas <- readLines(archivo)
  ok <- length(lineas) == n && all(grepl("^[0-9]+$", lineas)) && !anyDuplicated(lineas)
  cat(archivo, "-", length(lineas), "filas -", if (ok) "OK" else "ERROR", "\n")
  if (!ok) stop(paste("Problema en", archivo))
}
reloj("FV8 terminada: entregas listas")